# 🛒 Sample Superstore — Sales Data Analytics
## Data Analytics Internship Project

| Field | Detail |
|---|---|
| **Author** | Pranita Nande |
| **Dataset** | `samplesuperstore.csv` — 10,194 rows × 21 columns |
| **Period** | January 2023 – December 2026 |
| **Tools** | Python · pandas · matplotlib · seaborn |
| **Original CSV** | Never modified — all work on an in-memory copy |

---

### Project Sections
1. Load & Inspect  
2. Data Cleaning  
3. KPI Dashboard  
4. Category & Sub-Category Analysis  
5. Segment Analysis  
6. Regional Analysis  
7. Discount vs Profit Analysis  
8. Time-Series & Seasonality  
9. Customer & Product Analysis  
10. Risk Identification  
11. Opportunity Identification  
12. Business Recommendations (Fact → Insight → Action)  
13. Chart Gallery (12 charts)  

---
## Setup — Imports & Configuration

In [ ]:
# ── stdlib
import warnings
from pathlib import Path

# ── third-party
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

# ── design palette
BLUE   = '#3b82d4'
PURPLE = '#7c5cd8'
RED    = '#e53935'
GREEN  = '#1e8a4a'
AMBER  = '#f59e0b'
MUTED  = '#57606a'

sns.set_theme(style='whitegrid', font_scale=0.95)
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
})

CHART_DIR = Path('charts')
CHART_DIR.mkdir(exist_ok=True)

print('✔  Setup complete')

In [ ]:
# ── helper functions

def fmt_usd(v: float, decimals: int = 0) -> str:
    """Format a number as a compact USD string."""
    if abs(v) >= 1_000_000:
        return f'${v/1_000_000:.2f}M'
    if abs(v) >= 1_000:
        return f'${v/1_000:.1f}K'
    return f'${v:,.{decimals}f}'


def summarise(df: pd.DataFrame, group_col: str, sort_by: str = 'Sales') -> pd.DataFrame:
    """Aggregate Sales / Profit / Orders by one grouping column."""
    g = df.groupby(group_col, observed=True).agg(
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Order ID', 'nunique'),
        Rows=('Sales', 'count'),
    ).reset_index()
    g['Margin %'] = (g['Profit'] / g['Sales'] * 100).round(2)
    g['AOV']      = (g['Sales']  / g['Orders']).round(2)
    return g.sort_values(sort_by, ascending=False).reset_index(drop=True)


def save_fig(fig, name: str) -> str:
    path = str(CHART_DIR / f'{name}.png')
    fig.savefig(path, bbox_inches='tight', dpi=120)
    return path


print('✔  Helper functions defined')

---
## Section 1 — Load & Inspect

In [ ]:
RAW_FILE = 'samplesuperstore.csv'
df_raw = pd.read_csv(RAW_FILE, dtype={'Postal Code': str})

print(f'Shape         : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Missing values: {df_raw.isnull().sum().sum()}')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print(f'Date range    : {df_raw["Order Date"].min()} → {df_raw["Order Date"].max()}')

In [ ]:
# Data types
df_raw.dtypes.to_frame('dtype')

In [ ]:
# Numeric summary
df_raw[['Sales', 'Quantity', 'Discount', 'Profit']].describe().round(2)

In [ ]:
# Preview first 5 rows
df_raw.head()

---
## Section 2 — Data Cleaning
> The original CSV is never modified. All cleaning is done on `df`, an in-memory copy.

In [ ]:
df = df_raw.copy()

# 2a. Parse date columns
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')
print('✔  Dates parsed to datetime')

# 2b. Fix US ZIP codes stripped of leading zero  (438 rows affected)
us_mask = df['Country/Region'] == 'United States'
before  = (df.loc[us_mask, 'Postal Code'].str.len() < 5).sum()
df.loc[us_mask, 'Postal Code'] = df.loc[us_mask, 'Postal Code'].str.zfill(5)
print(f'✔  Fixed {before} US ZIP codes (leading zero restored)')

# 2c. Derived temporal columns
df['Order Year']  = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order YM']    = df['Order Date'].dt.to_period('M')
df['Ship Lag']    = (df['Ship Date'] - df['Order Date']).dt.days
print('✔  Added: Order Year, Order Month, Order YM, Ship Lag')

# 2d. Business flag columns
df['Is Loss'] = df['Profit'] < 0
df['Discount Bracket'] = pd.cut(
    df['Discount'],
    bins=[-0.001, 0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 1.0],
    labels=['0%', '1–10%', '11–20%', '21–30%', '31–40%', '41–50%', '51–80%'],
)
print('✔  Added: Is Loss, Discount Bracket')

# 2e. Save cleaned copy (original untouched)
df.to_csv('superstore_cleaned.csv', index=False)
print(f'\nCleaned dataset saved → superstore_cleaned.csv')
print(f'Shape after cleaning : {df.shape[0]:,} rows × {df.shape[1]} columns')

In [ ]:
# Cleaning summary table
cleaning_log = pd.DataFrame([
    ('Missing values',              0,      'None required',                        '✅ Clean'),
    ('Duplicate rows',              0,      'None required',                        '✅ Clean'),
    ('US ZIP leading-zero missing', 438,    'str.zfill(5) applied',                 '✅ Fixed'),
    ('Date columns stored as str',  10194,  'pd.to_datetime() applied',             '✅ Fixed'),
    ('Canadian postal codes short', 200,    'Partial FSA codes — flagged only',     '⚠️ Flagged'),
], columns=['Issue', 'Rows Affected', 'Action', 'Status'])
cleaning_log

---
## Section 3 — KPI Dashboard

In [ ]:
total_sales      = df['Sales'].sum()
total_profit     = df['Profit'].sum()
total_orders     = df['Order ID'].nunique()
total_customers  = df['Customer ID'].nunique()
total_products   = df['Product ID'].nunique()
aov              = total_sales / total_orders
profit_margin    = total_profit / total_sales * 100
avg_discount     = df['Discount'].mean() * 100
loss_rows        = int(df['Is Loss'].sum())
loss_pct         = loss_rows / len(df) * 100
avg_ship_lag     = df['Ship Lag'].mean()
repeat_customers = (df.groupby('Customer ID')['Order ID'].nunique() > 1).sum()
repeat_rate      = repeat_customers / total_customers * 100

kpi_df = pd.DataFrame([
    ('Total Sales',           fmt_usd(total_sales),                    '4-yr cumulative revenue'),
    ('Total Profit',          fmt_usd(total_profit),                   'After all discounts'),
    ('Profit Margin',         f'{profit_margin:.2f}%',                 'Net margin overall'),
    ('Total Orders',          f'{total_orders:,}',                     'Unique Order IDs'),
    ('Total Customers',       f'{total_customers:,}',                  'Unique Customer IDs'),
    ('Total Products',        f'{total_products:,}',                   'Unique Product IDs'),
    ('Avg Order Value (AOV)', f'${aov:,.2f}',                          'Sales ÷ Orders'),
    ('Avg Discount',          f'{avg_discount:.2f}%',                  'Margin compressor'),
    ('Loss Transactions',     f'{loss_rows:,} ({loss_pct:.1f}%)',      'Negative-profit rows'),
    ('Avg Ship Lag',          f'{avg_ship_lag:.1f} days',              'Order → Ship'),
    ('Repeat Customers',      f'{repeat_customers} ({repeat_rate:.1f}%)', '> 1 order placed'),
], columns=['KPI', 'Value', 'Note'])

kpi_df

---
## Section 4 — Category & Sub-Category Analysis

In [ ]:
cat_df = summarise(df, 'Category')
print('── By Category ──────────────────────────────────────────')
display(cat_df[['Category','Sales','Profit','Margin %','Orders','AOV']])

In [ ]:
sub_df = summarise(df, 'Sub-Category', 'Profit')
print('── By Sub-Category (sorted by Profit) ──────────────────')
display(sub_df[['Sub-Category','Sales','Profit','Margin %','Orders']])

loss_subcats = sub_df[sub_df['Profit'] < 0]['Sub-Category'].tolist()
print(f'\n⚠  Loss-making sub-categories: {loss_subcats}')

---
## Section 5 — Segment Analysis

In [ ]:
seg_df = summarise(df, 'Segment')
seg_cust = df.groupby('Segment')['Customer ID'].nunique().reset_index(name='Customers')
seg_df   = seg_df.merge(seg_cust, on='Segment')
seg_df['Profit/Customer'] = (seg_df['Profit'] / seg_df['Customers']).round(2)

display(seg_df[['Segment','Sales','Profit','Margin %','Orders','Customers','Profit/Customer']])

best_seg = seg_df.sort_values('Margin %', ascending=False).iloc[0]
print(f'\n✦  Highest-margin segment: {best_seg["Segment"]} at {best_seg["Margin %"]}%')

---
## Section 6 — Regional Analysis

In [ ]:
reg_df = summarise(df, 'Region')
print('── By Region ──────────────────────────────────────────')
display(reg_df[['Region','Sales','Profit','Margin %','Orders']])

state_df = summarise(df, 'State/Province')
print('\n── Top 10 States by Sales ──────────────────────────────')
display(state_df[['State/Province','Sales','Profit','Margin %']].head(10))

print('\n── Bottom 5 States by Profit ───────────────────────────')
display(state_df[['State/Province','Sales','Profit','Margin %']].tail(5))

---
## Section 7 — Discount vs Profit Analysis

In [ ]:
disc = (
    df.groupby('Discount Bracket', observed=True)
    .agg(
        Rows=('Sales', 'count'),
        Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Losses=('Is Loss', 'sum'),
    )
    .reset_index()
)
disc['Margin %']    = (disc['Profit'] / disc['Sales'] * 100).round(2)
disc['Loss Rate %'] = (disc['Losses'] / disc['Rows']  * 100).round(1)

display(disc[['Discount Bracket','Rows','Sales','Profit','Margin %','Loss Rate %']])

In [ ]:
# Correlation matrix
corr = df[['Sales','Discount','Profit','Quantity']].corr()
print('── Correlation Matrix ──────────────────────────────────')
display(corr.round(3))

disc_profit_corr = corr.loc['Discount','Profit']
print(f'\nDiscount ↔ Profit correlation: {disc_profit_corr:.3f}  '
      f'({"strong negative" if disc_profit_corr < -0.3 else "moderate negative"})')

---
## Section 8 — Time-Series & Seasonality

In [ ]:
annual = df.groupby('Order Year').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique'),
).reset_index()
annual['Margin %']     = (annual['Profit'] / annual['Sales'] * 100).round(2)
annual['Sales YoY %']  = annual['Sales'].pct_change().mul(100).round(2)
annual['Profit YoY %'] = annual['Profit'].pct_change().mul(100).round(2)

print('── Annual Trend ────────────────────────────────────────')
display(annual)

monthly_avg = (
    df.groupby('Order Month')['Sales'].sum() / df['Order Year'].nunique()
).reset_index(name='Avg Monthly Sales')
monthly_avg['Month'] = ['Jan','Feb','Mar','Apr','May','Jun',
                        'Jul','Aug','Sep','Oct','Nov','Dec']

peak_month = monthly_avg.sort_values('Avg Monthly Sales', ascending=False).iloc[0]
slow_month = monthly_avg.sort_values('Avg Monthly Sales').iloc[0]
print(f'\nPeak month  : {peak_month["Month"]}  (avg ${peak_month["Avg Monthly Sales"]:,.0f})')
print(f'Slowest month: {slow_month["Month"]}  (avg ${slow_month["Avg Monthly Sales"]:,.0f})')

---
## Section 9 — Customer & Product Analysis

In [ ]:
cust_df = summarise(df, 'Customer Name').head(10)
print('── Top 10 Customers by Sales ───────────────────────────')
display(cust_df[['Customer Name','Sales','Profit','Margin %','Orders']])

In [ ]:
prod_df = summarise(df, 'Product Name', 'Profit').head(10)
print('── Top 10 Products by Profit ───────────────────────────')
display(prod_df[['Product Name','Sales','Profit','Margin %']])

loss_prod_df = summarise(df, 'Product Name', 'Profit')
loss_prod_df = loss_prod_df[loss_prod_df['Profit'] < 0].head(10)
print('\n── Top 10 Loss-Making Products ─────────────────────────')
display(loss_prod_df[['Product Name','Sales','Profit','Margin %']])

---
## Section 10 — Risk Identification

In [ ]:
heavy   = df[df['Discount'] > 0.30]
r1_loss = heavy['Profit'].sum()
r2_loss = sub_df[sub_df['Profit'] < 0]['Profit'].sum()
central = df[df['Region'] == 'Central']
r3_margin = central['Profit'].sum() / central['Sales'].sum() * 100
top_cust   = summarise(df, 'Customer Name')
loss_cust  = top_cust[(top_cust['Sales'] > 10000) & (top_cust['Profit'] < 0)]
q4_sales   = df[df['Order Month'].isin([10, 11, 12])]['Sales'].sum()
q4_pct     = q4_sales / total_sales * 100

risks = pd.DataFrame([
    ('R1', 'HIGH',   'Excessive Discounting (>30%)',
     f'{len(heavy):,} orders, total profit destroyed: {fmt_usd(r1_loss)}'),
    ('R2', 'HIGH',   'Loss-Making Sub-Categories',
     f'Tables −$17,753 | Bookcases −$3,632 | Supplies −$1,171. Combined: {fmt_usd(r2_loss)}'),
    ('R3', 'MEDIUM', 'Central Region Underperformance',
     f'Central margin {r3_margin:.1f}% vs West 15%. TX/IL/OH lost $53K combined.'),
    ('R4', 'MEDIUM', 'High-Revenue Loss Customers',
     f'{len(loss_cust)} customer(s) with >$10K sales but negative profit.'),
    ('R5', 'LOW',    'Q4 Revenue Concentration',
     f'Q4 = {q4_pct:.1f}% of annual revenue; any disruption has outsized impact.'),
], columns=['ID', 'Severity', 'Risk', 'Detail'])

display(risks)

---
## Section 11 — Opportunity Identification

In [ ]:
ho      = df[df['Segment'] == 'Home Office']
ho_m    = ho['Profit'].sum() / ho['Sales'].sum() * 100
west    = df[df['Region'] == 'West']
west_m  = west['Profit'].sum() / west['Sales'].sum() * 100
top5_m  = sub_df.sort_values('Margin %', ascending=False).head(5)['Sub-Category'].tolist()

opps = pd.DataFrame([
    ('O1', 'Cap Discounts at 20%',
     'Recover ~$100K profit/yr. Zero-discount orders yield 29.6% margin.'),
    ('O2', 'Scale Copiers & Paper',
     f'Top 5 by margin: {", ".join(top5_m)}. 20% Copier volume uplift → +$11K profit.'),
    ('O3', f'Grow Home Office Segment ({ho_m:.1f}% margin)',
     'Highest margin segment, only 149 customers — significant headroom.'),
    ('O4', f'Apply West Playbook to Central ({west_m:.1f}% → 7.9%)',
     'Same product mix, different discount discipline. Worth ~$35K in profit.'),
    ('O5', 'Convert Q4 Buyers to Q1 Repeat Customers',
     'Loyalty programmes (not discounts) to smooth seasonal revenue trough.'),
], columns=['ID', 'Opportunity', 'Detail'])

display(opps)

---
## Section 12 — Business Recommendations
> Framework: **Fact → Insight → Action**

In [ ]:
recs = [
    {
        'no': 1, 'title': 'Implement a Hard Discount Cap at 20%',
        'fact':    'Every order above 30% discount is loss-making. 874 orders at 51–80% discount wiped $77K in profit.',
        'insight': 'Discounts above 30% do not recover through volume — they erode margin with zero sales lift.',
        'action':  'Set policy max discount 20%. Require VP sign-off above 25%. Expected recovery: $80–100K/yr.',
    },
    {
        'no': 2, 'title': 'Reprice or Discontinue Tables Sub-Category',
        'fact':    'Tables: $208K sales, −$17,753 profit (−8.5% margin). 88.9% of 31–40% discount orders lose money.',
        'insight': 'Cost structure cannot support current prices even before discounts.',
        'action':  'Raise base prices 15–20%, renegotiate supplier costs, or phase out bottom-5 SKUs by margin.',
    },
    {
        'no': 3, 'title': 'Fix Central Region Discount Discipline',
        'fact':    'Central margin 7.9% vs West 15%. TX, IL, OH together lost $53K.',
        'insight': 'Gap is discount-driven — same categories underperform in Central vs West.',
        'action':  'Audit Central rep discount authorisation. Apply West pricing playbook to Central team.',
    },
    {
        'no': 4, 'title': 'Grow the Home Office Segment',
        'fact':    'Home Office: 14.0% margin (highest), but only 149 customers out of 804.',
        'insight': 'Home Office buyers purchase with fewer bulk discounts → structurally better margins.',
        'action':  'Launch remote-worker acquisition campaign. Curated Tech + Office bundles at ≤15% discount.',
    },
    {
        'no': 5, 'title': 'Invest in Q4 Inventory; Run Q1 Loyalty Campaigns',
        'fact':    f'Q4 = {q4_pct:.0f}% of annual revenue. February is weakest month every year.',
        'insight': 'Seasonal demand is highly predictable. Q1 blanket discounts sacrifice margin for minimal volume.',
        'action':  'Stock +25–30% high-margin SKUs by August. Run Q1 loyalty (bundles/points), not price cuts.',
    },
    {
        'no': 6, 'title': 'Build a Customer Profitability Scorecard',
        'fact':    'Sean Miller (#1 by revenue $25K) generates −$1,981 profit.',
        'insight': 'Revenue-ranked lists hide loss-generating relationships.',
        'action':  'Quarterly review: Sales, Profit, Margin, Discount Rate per customer. Min-margin policy for offenders.',
    },
]

recs_df = pd.DataFrame(recs)[['no','title','fact','insight','action']]
recs_df.columns = ['#', 'Title', 'Fact', 'Insight', 'Action']
display(recs_df)

---
## Section 13 — Chart Gallery
All 12 charts are rendered inline and also saved to `charts/`.

In [ ]:
# C1 — Sales & Profit by Category
fig, ax = plt.subplots(figsize=(8, 4.5))
x, w = range(len(cat_df)), 0.35
ax.bar([i - w/2 for i in x], cat_df['Sales'],  width=w, color=BLUE,   label='Sales')
b2 = ax.bar([i + w/2 for i in x], cat_df['Profit'], width=w, color=PURPLE, label='Profit')
for bar in b2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 3000, fmt_usd(h),
            ha='center', va='bottom', fontsize=8, color=PURPLE)
ax.set_xticks(list(x)); ax.set_xticklabels(cat_df['Category'])
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
ax.set_title('Chart 1 — Sales & Profit by Category', fontweight='bold')
ax.legend(framealpha=0.5)
fig.tight_layout(); save_fig(fig, 'c1_category'); plt.show()

In [ ]:
# C2 — Profit by Sub-Category
sub_s  = summarise(df, 'Sub-Category', 'Profit')
colors = [RED if v < 0 else BLUE for v in sub_s['Profit']]
fig, ax = plt.subplots(figsize=(9, 6.5))
bars = ax.barh(sub_s['Sub-Category'], sub_s['Profit'], color=colors, edgecolor='white')
ax.axvline(0, color='#333', linewidth=0.9)
for bar, val in zip(bars, sub_s['Profit']):
    ax.text(val + (800 if val >= 0 else -800), bar.get_y() + bar.get_height()/2,
            fmt_usd(val), va='center', ha='left' if val >= 0 else 'right', fontsize=7.5)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.set_title('Chart 2 — Profit by Sub-Category', fontweight='bold')
ax.legend(handles=[mpatches.Patch(color=BLUE, label='Profit'),
                   mpatches.Patch(color=RED,  label='Loss')], loc='lower right')
fig.tight_layout(); save_fig(fig, 'c2_subcategory'); plt.show()

In [ ]:
# C3 — Discount Bracket vs Margin & Loss Rate
fig, ax1 = plt.subplots(figsize=(9, 4.5))
bar_c = [RED if v < 0 else BLUE for v in disc['Margin %']]
ax1.bar(disc['Discount Bracket'].astype(str), disc['Margin %'], color=bar_c, alpha=0.85, zorder=2)
ax1.axhline(0, color='#333', linewidth=0.8, zorder=3)
ax1.set_ylabel('Profit Margin (%)'); ax1.set_xlabel('Discount Bracket')
ax1.set_title('Chart 3 — Discount Bracket vs Profit Margin & Loss Rate', fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(disc['Discount Bracket'].astype(str), disc['Loss Rate %'],
         marker='o', color=RED, linestyle='--', linewidth=2, label='Loss Rate %', zorder=4)
ax2.set_ylabel('Loss Rate (%)', color=RED); ax2.tick_params(axis='y', labelcolor=RED)
ax2.set_ylim(0, 115); ax2.legend(loc='upper left')
fig.tight_layout(); save_fig(fig, 'c3_discount'); plt.show()

In [ ]:
# C4 — Monthly Sales Seasonality
monthly = df.groupby(['Order Year', 'Order Month'])['Sales'].sum().reset_index()
yc = {2023: '#aab8c2', 2024: PURPLE, 2025: BLUE, 2026: '#1a5ea8'}
fig, ax = plt.subplots(figsize=(12, 4.5))
for yr, grp in monthly.groupby('Order Year'):
    ax.plot(grp['Order Month'], grp['Sales'], marker='o', label=str(yr), color=yc[yr], linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.set_title('Chart 4 — Monthly Sales Trend by Year (Seasonality)', fontweight='bold')
ax.set_ylabel('Sales ($)'); ax.legend(title='Year')
fig.tight_layout(); save_fig(fig, 'c4_seasonality'); plt.show()

In [ ]:
# C5 — Annual Sales & Profit Trend
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(annual['Order Year'], annual['Sales'],  marker='s', color=BLUE,   label='Sales',  linewidth=2.2)
ax.plot(annual['Order Year'], annual['Profit'], marker='s', color=PURPLE, label='Profit', linewidth=2.2)
for _, row in annual.iterrows():
    ax.annotate(fmt_usd(row['Sales']),  (row['Order Year'], row['Sales']),
                textcoords='offset points', xytext=(0, 8),  ha='center', fontsize=8, color=BLUE)
    ax.annotate(fmt_usd(row['Profit']), (row['Order Year'], row['Profit']),
                textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8, color=PURPLE)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.set_title('Chart 5 — Annual Sales & Profit Trend (2023–2026)', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Amount ($)'); ax.legend()
fig.tight_layout(); save_fig(fig, 'c5_annual'); plt.show()

In [ ]:
# C6 — Segment: Donut + Margin
seg_s = summarise(df, 'Segment')
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].pie(seg_s['Sales'], labels=seg_s['Segment'], autopct='%1.1f%%', startangle=90,
            colors=[BLUE, PURPLE, '#57a0d3'], wedgeprops={'width': 0.55})
axes[0].set_title('Sales Share by Segment', fontweight='bold')
axes[1].bar(seg_s['Segment'], seg_s['Margin %'], color=[BLUE, PURPLE, '#57a0d3'])
axes[1].set_title('Profit Margin by Segment', fontweight='bold'); axes[1].set_ylabel('Margin (%)')
for i, (_, row) in enumerate(seg_s.iterrows()):
    axes[1].text(i, row['Margin %'] + 0.1, f"{row['Margin %']:.1f}%", ha='center', fontsize=9)
fig.suptitle('Chart 6 — Customer Segment Analysis', fontweight='bold')
fig.tight_layout(); save_fig(fig, 'c6_segment'); plt.show()

In [ ]:
# C7 — Region Sales & Profit
reg_s = summarise(df, 'Region')
fig, ax = plt.subplots(figsize=(8, 4.5))
x = range(len(reg_s))
ax.bar([i - 0.2 for i in x], reg_s['Sales'],  width=0.38, color=BLUE,   label='Sales')
ax.bar([i + 0.2 for i in x], reg_s['Profit'], width=0.38, color=PURPLE, label='Profit')
ax.set_xticks(list(x)); ax.set_xticklabels(reg_s['Region'])
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.set_title('Chart 7 — Sales & Profit by Region', fontweight='bold'); ax.legend()
for i, (_, row) in enumerate(reg_s.iterrows()):
    ax.text(i, row['Sales'] + 5000, f"Margin\n{row['Margin %']:.1f}%", ha='center', fontsize=7.5, color=MUTED)
fig.tight_layout(); save_fig(fig, 'c7_region'); plt.show()

In [ ]:
# C8 — Top 5 & Bottom 5 States by Profit
sp = summarise(df, 'State/Province', 'Profit')
combined = pd.concat([sp.head(5), sp.tail(5)]).reset_index(drop=True)
cs = [GREEN if v >= 0 else RED for v in combined['Profit']]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(combined['State/Province'], combined['Profit'], color=cs)
ax.axhline(0, color='#333', linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.set_title('Chart 8 — Top 5 & Bottom 5 States by Profit', fontweight='bold')
ax.set_ylabel('Profit ($)'); plt.xticks(rotation=30, ha='right', fontsize=8)
fig.tight_layout(); save_fig(fig, 'c8_states'); plt.show()

In [ ]:
# C9 — Top 10 Products by Profit
tp = summarise(df, 'Product Name', 'Profit').head(10)
tp['Short'] = tp['Product Name'].str[:35] + '…'
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(tp['Short'][::-1], tp['Profit'][::-1], color=BLUE)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.set_title('Chart 9 — Top 10 Products by Profit', fontweight='bold')
ax.set_xlabel('Profit ($)')
fig.tight_layout(); save_fig(fig, 'c9_top_products'); plt.show()

In [ ]:
# C10 — Top 10 Loss-Making Products
lp = summarise(df, 'Product Name', 'Profit')
lp = lp[lp['Profit'] < 0].head(10)
lp['Short'] = lp['Product Name'].str[:35] + '…'
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(lp['Short'][::-1], lp['Profit'][::-1], color=RED)
ax.axvline(0, color='#333', linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.1f}K'))
ax.set_title('Chart 10 — Top 10 Loss-Making Products', fontweight='bold')
ax.set_xlabel('Profit ($)')
fig.tight_layout(); save_fig(fig, 'c10_loss_products'); plt.show()

In [ ]:
# C11 — Ship Mode: Sales & Margin
ship_df = summarise(df, 'Ship Mode')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(ship_df['Ship Mode'], ship_df['Sales'], color=BLUE)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
axes[0].set_title('Sales by Ship Mode', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(ship_df['Ship Mode'], ship_df['Margin %'], color=PURPLE)
axes[1].set_title('Profit Margin by Ship Mode', fontweight='bold')
axes[1].set_ylabel('Margin (%)'); axes[1].tick_params(axis='x', rotation=20)
for i, (_, row) in enumerate(ship_df.iterrows()):
    axes[1].text(i, row['Margin %'] + 0.1, f"{row['Margin %']:.1f}%", ha='center', fontsize=9)
fig.suptitle('Chart 11 — Ship Mode Analysis', fontweight='bold')
fig.tight_layout(); save_fig(fig, 'c11_shipmode'); plt.show()

In [ ]:
# C12 — Discount vs Profit Scatter
sample = df.sample(min(2000, len(df)), random_state=42)
sc = [RED if p < 0 else BLUE for p in sample['Profit']]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(sample['Discount'], sample['Profit'], c=sc, alpha=0.35, s=18, linewidths=0)
ax.axhline(0, color='#333', linewidth=0.8)
ax.axvline(0.30, color=AMBER, linewidth=1.5, linestyle='--', label='30% threshold')
ax.set_xlabel('Discount Rate'); ax.set_ylabel('Profit ($)')
ax.set_title('Chart 12 — Discount vs Profit (2,000 sample)', fontweight='bold')
ax.legend(handles=[mpatches.Patch(color=BLUE, label='Profit'),
                   mpatches.Patch(color=RED, label='Loss'),
                   plt.Line2D([0],[0], color=AMBER, linestyle='--', label='30% threshold')])
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
fig.tight_layout(); save_fig(fig, 'c12_scatter'); plt.show()

---
## Summary

| KPI | Value |
|---|---|
| Total Sales (4-yr) | **$2.33M** |
| Total Profit | **$292K** |
| Profit Margin | **12.56%** |
| Sales Growth (2023→2026) | **+51%** |
| Profit Growth (2023→2026) | **+86%** |
| Loss Transactions | **1,901 (18.6%)** |

### Top 3 Business Actions
1. **Cap discounts at 20%** — recovers $80–100K/yr in profit immediately
2. **Reprice Tables sub-category** — eliminates the largest single source of loss (−$17,753/yr)
3. **Grow Home Office segment** — highest-margin (14%), fewest customers (149), biggest growth lever

---
*Project by Pranita Nande — Data Analytics Internship*